# Train a tiny character-level GPT

This notebook trains a very small transformer (~0.8M parameters) on the **Tiny Shakespeare** dataset. It's designed to finish in a few minutes on a free Colab GPU, and produces a checkpoint you can plug into the FastAPI backend.

**Steps:** enable GPU (`Runtime > Change runtime type > T4 GPU`), then run all cells top to bottom.

Dataset (Kaggle): https://www.kaggle.com/datasets/thedevastator/the-bards-best-a-character-modeling-dataset

## 1. Install dependencies

In [ ]:
!pip install -q kaggle

## 2. Get the dataset

**Option A — Kaggle API (recommended, uses the real Kaggle dataset link above).**
1. Go to kaggle.com -> your profile picture -> Settings -> API -> "Create New Token". This downloads `kaggle.json`.
2. Run the cell below and upload that file when prompted.

**Option B — no Kaggle account needed.** Skip to the cell after and it will fall back to a mirrored copy of the same text (identical Tiny Shakespeare corpus) if no `kaggle.json` is found.

In [ ]:
import os
from google.colab import files

os.makedirs('/root/.kaggle', exist_ok=True)
print('Upload your kaggle.json (or skip this cell to use the fallback download instead)')
try:
    uploaded = files.upload()
    for fname in uploaded:
        os.rename(fname, '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    HAVE_KAGGLE_CREDS = True
except Exception as e:
    print('No file uploaded, will use fallback in the next cell.', e)
    HAVE_KAGGLE_CREDS = False

In [ ]:
import os, glob

text = None
if HAVE_KAGGLE_CREDS:
    !kaggle datasets download -d thedevastator/the-bards-best-a-character-modeling-dataset -p /content/data --unzip
    csv_or_txt = glob.glob('/content/data/*')
    print('Downloaded files:', csv_or_txt)
    # The Kaggle dataset ships as CSV; concatenate the text column into one corpus.
    import pandas as pd
    for f in csv_or_txt:
        if f.endswith('.csv'):
            df = pd.read_csv(f)
            text_col = [c for c in df.columns if df[c].dtype == object][0]
            text = '\n'.join(df[text_col].astype(str).tolist())
            break

if text is None:
    # Fallback: same Tiny Shakespeare corpus, no auth required.
    !wget -q -O /content/input.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
    text = open('/content/input.txt').read()

print(f'Corpus length: {len(text):,} characters')
print(text[:300])

## 3. Model definition
(This must stay identical to `backend/model.py` in the repo — copy this cell there if you change it.)

In [ ]:
import math
import torch
import torch.nn as nn
from torch.nn import functional as F

class ModelConfig:
    def __init__(self, vocab_size, block_size=128, n_layer=4, n_head=4, n_embd=128, dropout=0.1):
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.dropout = dropout

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head
        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.attn_drop = nn.Dropout(config.dropout)
        self.resid_drop = nn.Dropout(config.dropout)
        mask = torch.tril(torch.ones(config.block_size, config.block_size))
        self.register_buffer('mask', mask.view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)
        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(out))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.pos_emb = nn.Parameter(torch.zeros(1, config.block_size, config.n_embd))
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.Sequential(*[Block(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.tok_emb(idx)
        pos = self.pos_emb[:, :T, :]
        x = self.drop(tok + pos)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_id), dim=1)
        return idx

print('Model classes defined.')

## 4. Build the character vocabulary and dataset tensors

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join(itos[i] for i in l)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f'Vocab size: {vocab_size}, train tokens: {len(train_data):,}, val tokens: {len(val_data):,}')

## 5. Train

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

block_size = 128
batch_size = 64
max_iters = 2000
eval_interval = 200
learning_rate = 3e-4

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(model, eval_iters=50):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

config = ModelConfig(vocab_size=vocab_size, block_size=block_size, n_layer=4, n_head=4, n_embd=128, dropout=0.1)
model = TinyGPT(config).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model has {n_params:,} parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for it in range(max_iters):
    if it % eval_interval == 0 or it == max_iters - 1:
        losses = estimate_loss(model)
        print(f"step {it}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print('Training done.')

## 6. Try generating some text

In [ ]:
context = torch.tensor([encode('ROMEO:')], dtype=torch.long, device=device)
generated = model.generate(context, max_new_tokens=300, temperature=0.8, top_k=40)
print(decode(generated[0].tolist()))

## 7. Save the checkpoint

This saves `tiny_llm.pt` (weights + config + vocab) and downloads it to your computer. Put the downloaded file into `backend/checkpoints/tiny_llm.pt` in your GitHub repo.

In [ ]:
checkpoint = {
    'model_state_dict': model.state_dict(),
    'config': vars(config),
    'stoi': stoi,
    'itos': itos,
}
torch.save(checkpoint, '/content/tiny_llm.pt')
files.download('/content/tiny_llm.pt')